In [1]:
import yaml
from utils import set_seed, load_graph_data

with open(r'C:/Users/Damir/Desktop/GNN_dataset_test/GNN_dataset_test/params.yaml') as f:
    cfg = yaml.safe_load(f)

configs_model = cfg["model"]
configs_train = cfg["train"]
configs_paths = cfg["paths"]
configs_preproc = cfg["preprocess"]

set_seed(configs_train["seed"])

train_dataloader, val_dataloader, data_features = load_graph_data(configs_paths, configs_preproc, configs_train, return_val = True)

hetero_graph_batch, sample_names = next(iter(train_dataloader))
hetero_graph_batch, sample_names

c:\Users\Damir\Desktop\GNN_dataset_test\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Метаданные (train часть):
       MODEL    STATUS                                                MIN  \
0  e1_v00008  COMPLETE  [6.5991211e+03 2.5237122e+00 1.7128232e+00 9.3...   
1  e1_v00004  COMPLETE  [6.5972900e+03 3.6298862e+00 2.0998237e+00 1.7...   
2  e1_v00007  COMPLETE  [6.5972900e+03 4.4991727e+00 2.8879199e+00 1.9...   
3  e1_v00001  COMPLETE  [6.5972900e+03 5.1691298e+00 3.9185262e+00 1.8...   
4  e1_v00005  COMPLETE  [6.5991211e+03 3.1294053e+00 3.4316871e+00 2.4...   
5  e1_v00002  COMPLETE  [6.5991211e+03 5.0439239e+00 5.2062578e+00 1.8...   
6  e1_v00010  COMPLETE  [6.5972900e+03 5.8906302e+00 4.2009263e+00 3.9...   
7  e1_v00009  COMPLETE  [6.5502930e+03 3.0328267e+00 1.9648471e+00 1.3...   
8  e1_v00006  COMPLETE  [6.5502930e+03 1.5860901e+00 1.9193324e+00 2.4...   
9  e1_v00003  COMPLETE  [6.5502930e+03 2.2270913e+00 1.2137179e+00 4.8...   

                                                 MAX  \
0  [1.0431519e+04 1.9569319e+02 1.3281540e+02 7.2...   
1  [1.0431519e

(HeteroDataBatch(
   cell={
     x=[12960, 11],
     labels=[12960],
     batch=[12960],
     ptr=[3],
   },
   well={
     x=[21, 1],
     y=[21, 3, 25],
     batch=[21],
     ptr=[3],
   },
   (cell, flows_to, cell)={
     edge_index=[2, 66992],
     edge_attr=[66992, 1],
   },
   (cell, linked_to, well)={ edge_index=[2, 355] }
 ),
 ('e1_v00008.pt', 'e1_v00003.pt'))

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cpu
False


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.5.1+cu121
True


In [2]:
import torch
import yaml
import os
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader

# Импортируем твою модель и функции
from simple_model import SimpleHeteroGNN
from utils import load_graph_data

# 1. Загрузка параметров из yaml
with open("params.yaml", "r") as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Работаем на: {device}")

# 2. Загрузка данных через твою функцию
# Она сама сделает split, посчитает статистики и нормализует всё
train_loader, val_loader, feat_list = load_graph_data(
    config['paths'], 
    config['preprocess'], 
    config['train']
)

# 3. Инициализация модели
# Берем один пример из базы, чтобы узнать размерности входных данных
sample_data, _ = next(iter(train_loader))

model = SimpleHeteroGNN(
    cell_features=sample_data['cell'].x.size(1),
    well_features=sample_data['well'].x.size(1),
    hidden_dim=config['model']['nz'], # используем nz из yaml
    out_seq_len=25,
    num_phases=3
).to(device)

Работаем на: cpu
Метаданные (train часть):
       MODEL    STATUS                                                MIN  \
0  e1_v00004  COMPLETE  [2.0998237e+00 3.0059695e-01 0.0000000e+00 2.9...   
1  e1_v00001  COMPLETE  [3.9185262e+00 3.0256003e-01 0.0000000e+00 2.9...   
2  e1_v00005  COMPLETE  [3.4316871e+00 3.0619934e-01 0.0000000e+00 2.9...   
3  e1_v00008  COMPLETE  [1.7128232e+00 3.1039816e-01 0.0000000e+00 2.9...   
4  e1_v00007  COMPLETE  [2.8879199e+00 3.0111608e-01 0.0000000e+00 2.9...   
5  e1_v00002  COMPLETE  [5.2062578e+00 3.0502006e-01 0.0000000e+00 2.9...   
6  e1_v00010  COMPLETE  [4.2009263e+00 3.0328861e-01 0.0000000e+00 2.9...   
7  e1_v00009  COMPLETE  [1.9648471e+00 3.0391216e-01 0.0000000e+00 2.9...   
8  e1_v00006  COMPLETE  [1.9193324e+00 3.0538654e-01 0.0000000e+00 2.9...   
9  e1_v00003  COMPLETE  [1.2137179e+00 3.0510870e-01 0.0000000e+00 2.9...   

                                                 MAX  \
0  [1.3609212e+02 1.0000000e+00 6.9940305e-01 2.9... 

c:\Users\Damir\Desktop\GNN_dataset_test\GNN_dataset_test\simple_model.py:25: UserWarning: There exist node types ({'cell'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  self.well_conv = HeteroConv({


In [3]:
if hasattr(sample_data['cell', 'flows_to', 'cell'], 'edge_attr'):
    print("edge_attr присутствует для cell->cell")
    edge_attr = sample_data['cell', 'flows_to', 'cell'].edge_attr
    print("Форма edge_attr:", edge_attr.shape)
else:
    print("edge_attr ОТСУТСТВУЕТ для cell->cell")

edge_attr присутствует для cell->cell
Форма edge_attr: torch.Size([25485, 1])


In [4]:
print("Атрибуты cell->cell:", sample_data['cell', 'flows_to', 'cell'].keys())

Атрибуты cell->cell: KeysView({'edge_index': tensor([[    0,     0,     1,  ..., 12956, 12957, 12958],
        [    1,    24,     2,  ..., 12957, 12958, 12959]]), 'edge_attr': tensor([[-0.5780],
        [-0.5749],
        [-0.8683],
        ...,
        [ 0.6223],
        [ 0.7204],
        [ 0.8352]])})


In [5]:
sample_data

HeteroDataBatch(
  cell={
    x=[12960, 8],
    labels=[12960],
    batch=[12960],
    ptr=[3],
  },
  well={
    x=[21, 1],
    y=[21, 3, 25],
    batch=[21],
    ptr=[3],
  },
  (cell, flows_to, cell)={
    edge_index=[2, 25485],
    edge_attr=[25485, 1],
  },
  (cell, linked_to, well)={ edge_index=[2, 355] }
)

In [6]:
edge_attr_cell = sample_data['cell', 'flows_to', 'cell'].get('edge_attr')
if edge_attr_cell is not None:
    print("edge_attr найден!")
else:
    print("edge_attr отсутствует")
edge_attr_well = sample_data['cell', 'linked_to', 'well'].get('edge_attr')
if edge_attr_well is not None:
    print("edge_attr найден!")
else:
    print("edge_attr отсутствует")

edge_attr найден!
edge_attr отсутствует


In [7]:
sample_data['cell'].x[sample_data['cell'].batch == 0].shape

torch.Size([6480, 8])

In [8]:
model(sample_data).shape

torch.Size([21, 3, 25])

In [9]:
sample_data['cell', 'flows_to', 'cell'].edge_index
sample_data['cell', 'linked_to', 'well'].edge_index

tensor([[  105,   969,  1401,  1833,  2697,  3129,  3993,  4425,  5289,  5721,
           225,   657,  1089,  1953,  2385,  3249,  4113,  4545,  4977,  5409,
           110,   542,   974,  1406,  1838,  3134,  3566,  3998,  4430,  4862,
          5294,  5726,  6158,   350,   782,  1214,  1646,  2942,  3374,  3806,
          4238,  5102,  5534,  6398,   230,  1094,  1526,  1958,  2390,  2822,
          3254,  4118,  5414,  6665,  7097,  7529,  7961,  8393,  8825,  9257,
          9689, 10121, 10553, 10985, 11417, 11849, 12281, 12713,  6590,  7022,
          7454,  7886,  8318,  8750,  9614, 10046, 10478, 10910, 11342, 11774,
         12206, 12638,  6737,  7169,  7601,  8033,  8465,  8897,  9329,  9761,
         10193, 10625, 11057, 11489, 12353, 12785,  6734,  7166,  7598,  8030,
          8462,  8894,  9326,  9758, 10622, 11054, 11486, 11918, 12782,  6830,
          7262,  7694,  8126,  8990,  9422,  9854, 10286, 10718, 11582, 12014,
         12878,  6826,  7690,  8122,  8554,  8986,  

In [12]:
sample_data['cell', 'linked_to', 'well'].edge_index

tensor([[  105,   969,  1401,  1833,  2697,  3129,  3993,  4425,  5289,  5721,
           225,   657,  1089,  1953,  2385,  3249,  4113,  4545,  4977,  5409,
           110,   542,   974,  1406,  1838,  3134,  3566,  3998,  4430,  4862,
          5294,  5726,  6158,   350,   782,  1214,  1646,  2942,  3374,  3806,
          4238,  5102,  5534,  6398,   230,  1094,  1526,  1958,  2390,  2822,
          3254,  4118,  5414,  4679,  5111,  5543,  5975,  6407,  6839,  7271,
          7703,  8135,  8567,  8999,  9431,  9863, 10295, 10727,  4604,  5036,
          5468,  5900,  6332,  6764,  7628,  8060,  8492,  8924,  9356,  9788,
         10220, 10652,  4751,  5183,  5615,  6047,  6479,  6911,  7343,  7775,
          8207,  8639,  9071,  9503, 10367, 10799,  4748,  5180,  5612,  6044,
          6476,  6908,  7340,  7772,  8636,  9068,  9500,  9932, 10796,  4844,
          5276,  5708,  6140,  7004,  7436,  7868,  8300,  8732,  9596, 10028,
         10892,  4840,  5704,  6136,  6568,  7000,  

In [10]:
x_dict = {
    'cell': model.cell_emb(data['cell'].x),
    'well': model.well_emb(data['well'].x)
}

# Связи
c2c_edge_index = data['cell', 'flows_to', 'cell'].edge_index
edge_index_dict = data.edge_index_dict

# Шаг 1: Пропаганда внутри пласта (ячейка -> ячейка)
h_cell = x_dict['cell']
for conv in model.cell_convs:
    h_cell = conv(h_cell, c2c_edge_index)
    h_cell = F.relu(h_cell)

x_dict['cell'] = h_cell

# Шаг 2: Сбор информации на скважины (ячейка -> скважина)
# well_updates получит эмбеддинги для узлов-приемников (скважин)
well_updates = model.well_conv(x_dict, edge_index_dict)
h_well = well_updates['well']

# Шаг 3: Финальный слой
out = model.well_mlp(h_well)
out.view(-1, 3, 25)

NameError: name 'data' is not defined

In [12]:
import plotly.graph_objects as go
import networkx as nx
import plotly.io as pio
pio.renderers.default = "browser"

def plot_graph_in_3d(hetero_graph_batch, data_features, color = 'well'):

    G = nx.Graph()

    # позиции ячеек (центры)
    positions = {
        i: (hetero_graph_batch['cell'].x[:, [data_features.index('X'), data_features.index('Y'), data_features.index('Z')]][i]*-1)
        for i in range(len(hetero_graph_batch['cell'].x)) 
    }

    positions 

    # добавляем вершины в граф
    for node, pos in positions.items():
        G.add_node(node, pos=list(pos))

    # рёбра
    G.add_edges_from(hetero_graph_batch[('cell', 'flows_to', 'cell')].edge_index.T.detach().numpy())

    # координаты вершин
    xs, ys, zs = zip(*[G.nodes[n]['pos'] for n in G.nodes()])
    if color == 'well':
        cs = [1 if n in hetero_graph_batch[('cell', 'linked_to', 'well')].edge_index[0] else 0 for n in G.nodes()]
    else:
        cs = hetero_graph_batch['cell'].x[:, data_features.index(color)]
    # Scatter3d с подписями
    node_trace = go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='markers',
        marker=dict(
            size=3.5,
            color=cs,#'lightsteelblue',
            line=dict(width=1, color='black'),
            colorscale='Viridis',   # Можно менять цветовую карту
            colorbar=dict(title=color),
        ),
 )

    # рёбра
    edge_x = []
    edge_y = []
    edge_z = []
    edge_color = []

    
    for i, (u, v) in enumerate(hetero_graph_batch[('cell', 'flows_to', 'cell')].edge_index.T.detach().numpy()):
        x0, y0, z0 = positions[u]
        x1, y1, z1 = positions[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
        edge_z += [z0, z1, None]

    edge_trace = go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode='lines',
        name = '',
        line=dict(width=5,
                color=edge_color,
                ),
    )

    # финальная фигура
    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        scene=dict(
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        ),
        showlegend=False,
        margin=dict(l=0, r=0, t=0, b=0))

    fig.show()


plot_graph_in_3d(hetero_graph_batch, data_features, color = 'PORO')

ModuleNotFoundError: No module named 'plotly'